# Uncertainty quantification with application to a tsunami case

## Load modules

In [1]:
# load specific functions for manipulating, plotting data
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as sc

# load specific functions of sklearn
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_error
from sklearn import model_selection
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Import all regression models you want to test
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

## Context

The application test case is adapted from the Ligurian (North West of Italy) 1887 tsunami case in particular studied by [Larroque et al. (2012)](https://academic.oup.com/gji/article/190/1/87/601785). The following figure provides the structural scheme of the fault rupture mechanism.

![ligurianEarthquake.png](https://raw.githubusercontent.com/anrhouses/TD_MLUQ/refs/heads/main/ligurianEarthquake.png).

We will focus on the impact of the uncertainties of the earthquake source parameters on the the sea surface elevatio denoted *SSH* induced by the tsunami. Nine uncertain parameters are considered:
* $X_{1} = Lon$, Longitude of the  epicentral position (°)
* $X_{2} = Lat$, Latitude of the  epicentral position (°)
* $X_{3} = Z$, Depth  (m)
* $X_{4} = Azi$, Azimuth  (°)
* $X_{5} = Dip$, Dip angle  (m)
* $X_{6} = Rake$, Rake angle  (°)
* $X_{7} = D$, average slip on the fault surface  (m)
* $X_{8} = W$, Width of the fault surface  (m)
* $X_{9} = L$, Length of the fault surface  (m)

A numerical model has been set up to estimate the tsunami-induced sea surface elevation at different locations along the French and Italian coast from the values of the earthquake source parameters. See details in [Rohmer et al. 2018](https://link.springer.com/article/10.1007/s00477-017-1423-y).
The model has $8$ input random variables. A set of 300 random samples was generated by assuming that all input variables vary uniformly between the lower and upper bound described in the Table below.
![Table](https://raw.githubusercontent.com/anrhouses/TD_MLUQ/refs/heads/main/Assumption.png)
For each input sample, a numerical model is run to compute the response (sea surface elevation) at all points along the coast.
![location.png](https://raw.githubusercontent.com/anrhouses/TD_MLUQ/refs/heads/main/location.png)

## Objectives
The objective of this tutorial is to illustrate the main steps of the framework for quantitative uncertainty management as described by [de Rocquigny et al. (2008)](https://onlinelibrary.wiley.com/doi/book/10.1002/9780470770733), namely:

![framework.png](https://raw.githubusercontent.com/anrhouses/TD_MLUQ/refs/heads/main/framework.png)

In this tutorial, you will perform:

- **Uncertainty propagation**
    - Propagate the uncertainty related to the source parameters of the earthquake
    - Estimate the percentile of the sea surface elevation
- **Sensitivity analysis**
    - Quantify the contribution of the source parameters of the earthquake to the uncertainty of sea surface elevation
    - Identify which source parameter(s) influence(s) the most the sea surface elevation

## Data

In [36]:
##Load the inputs
df = pd.read_csv("https://raw.githubusercontent.com/anrhouses/TD_MLUQ/refs/heads/main/DOE_aziPetit.txt",sep=" ")
X = df.values
d = df.shape[1] # number of inputs
N = df.shape[0] # number of samples

In [37]:
##Load the response (output)
outputSample0  = pd.read_csv("https://raw.githubusercontent.com/anrhouses/TD_MLUQ/refs/heads/main/Hmax_aziPetit.txt",sep=" ", header=None)

We wil first focus on location #8 =  Imperia (see map above).

In [38]:
loc = 8
Y = outputSample0.values[:,loc] ##output 
df['Y'] = Y

## Uncertainty propagation

We will compute the *alpha*-percentile of sea surface elevation denoted *Y* by running the following code. 

**QUESTION 1** Do the analysis for *alpha* = 0.75 with *N* = 300 observations, then for *alpha* = 0.95 and *alpha* = 0.975. Replace in the following ###. What can you see?

In [ ]:
## CHOOSE HERE
alpha = ###
N = ###

In [ ]:
print("Quantile at", alpha*100, "%=",sc.quantile(Y[:N], alpha))

**QUESTION 2** Do the same analysis but with *N* = 200 observations, then for *N* = 100. What can you see?

## Sensitivity analysis

**QUESTION 3** We will first analyse the input-output relationship by plotting the scatterplots. What can you say on the different relationships 

In [ ]:
fig, axs = plt.subplots(3,3)
axs[0,0].plot(df['Lon'].values, Y, 'o', label='', markerfacecolor='darkorange', markeredgecolor='black', alpha=0.8)
axs[0,0].set_title(['Lon'," vs ", "Y"])
axs[0,1].plot(df['Lat'].values, Y, 'o', label='', markerfacecolor='darkorange', markeredgecolor='black', alpha=0.8)
axs[0,1].set_title(['Lat'," vs ", "Y"])
axs[0,2].plot(df['Z'].values, Y, 'o', label='', markerfacecolor='darkorange', markeredgecolor='black', alpha=0.8)
axs[0,2].set_title(['Z'," vs ", "Y"])
axs[1,0].plot(df['Azi'].values, Y, 'o', label='', markerfacecolor='darkorange', markeredgecolor='black', alpha=0.8)
axs[1,0].set_title(['Azi'," vs ", "Y"])
axs[1,1].plot(df['Dip'].values, Y, 'o', label='', markerfacecolor='darkorange', markeredgecolor='black', alpha=0.8)
axs[1,1].set_title(['Dip'," vs ", "Y"])
axs[1,2].plot(df['rake'].values, Y, 'o', label='', markerfacecolor='darkorange', markeredgecolor='black', alpha=0.8)
axs[1,2].set_title(['rake'," vs ", "Y"])
axs[2,0].plot(df['L'].values, Y, 'o', label='', markerfacecolor='darkorange', markeredgecolor='black', alpha=0.8)
axs[2,0].set_title(['L'," vs ", "Y"])
axs[2,1].plot(df['W'].values, Y, 'o', label='', markerfacecolor='darkorange', markeredgecolor='black', alpha=0.8)
axs[2,1].set_title(['W'," vs ", "Y"])
axs[2,2].plot(df['D'].values, Y, 'o', label='', markerfacecolor='darkorange', markeredgecolor='black', alpha=0.8)
axs[2,2].set_title(['D'," vs ", "Y"])
fig.tight_layout()

**QUESTION 4** Can you identify some types of relation input-output? (linear, monotonic, etc.). Quantify them by running the correlation analysis below.

In [127]:
def feature_rank_plot(pred,metric,mmin,mmax,nominal,title,ylabel,mask): # feature ranking plot
    mpred = len(pred); mask_low = nominal-mask*(nominal-mmin); mask_high = nominal+mask*(mmax-nominal)
    plt.plot(pred,metric,color='black',zorder=20)
    plt.scatter(pred,metric,marker='o',s=10,color='black',zorder=100)
    plt.fill_between(np.arange(0,mpred,1),np.zeros(mpred),metric,where=(metric < nominal),interpolate=True,color='dodgerblue',alpha=0.3)
    plt.fill_between(np.arange(0,mpred,1),np.zeros(mpred),metric,where=(metric > nominal),interpolate=True,color='lightcoral',alpha=0.3)
    plt.fill_between(np.arange(0,mpred,1),np.full(mpred,mask_low),metric,where=(metric < mask_low),interpolate=True,color='blue',alpha=0.8,zorder=10)
    plt.fill_between(np.arange(0,mpred,1),np.full(mpred,mask_high),metric,where=(metric > mask_high),interpolate=True,color='red',alpha=0.8,zorder=10)  
    plt.xlabel('Predictor Features'); plt.ylabel(ylabel); plt.title(title)
    return

In [ ]:
pred = df.columns.values.tolist()[:9]
correlation = df.iloc[:,0:10].corr().iloc[9,:9]              # calculate correlation matrix and slice for only pred - resp
feature_rank_plot(pred,correlation,-1.0,1.0,0.0,'Feature Ranking, Correlation with ' + 'sea surface elevation','Correlation',0.5)
print("correlation coefficients = ", correlation)

**QUESTION 7** Do the same correlation analysis with the Spearman correlation coefficients. What can you see?

In [ ]:
rank_correlation, rank_correlation_pval = sc.spearmanr(df.iloc[:,0:10]) # calculate the range correlation coefficient
rank_correlation = rank_correlation[9,:9]
feature_rank_plot(pred,rank_correlation,-1.0,1.0,0.0,'Feature Ranking, Correlation with ' + 'sea surface elevation','Correlation',0.5)
print("rank correlation coefficients = ", rank_correlation)

**QUESTION 6** Run now a global sensitivity anlysis based on the first order Sobol' indices. We use here the rnak-based algorithm of [Gamboa et al. (2022)](https://projecteuclid.org/journals/bernoulli/volume-28/issue-4/Global-sensitivity-analysis--A-novel-generation-of-mighty-estimators/10.3150/21-BEJ1421.full). 
What can you say about the main effects of the input variables? Compare to the correlation analysis? Can you comment on the bootstrap-based confidence intervals also?

In [230]:
# Création de la fonction pour le calcul des estimateurs des rangs des indices de Sobol 
def compSRank(Y, X, I):
    """
    I est un indice de {1,...,p}
    X est une matrice de N lignes, chaque ligne a pour valeur (X1, X2, ..., Xp)
    Y est une la sortie
    """
    # Ordre selon X
    IS = np.argsort(X[: , I],axis = 0);
    
    # Sorties Y réordonnées selon X
    YRank = np.copy(Y)
    YRank = Y[IS]     
    
    # Sorties shiftées
    YShift = np.roll(YRank,-1)

    # Calcul de l'estimation de l'indice de Sobol
    Cov = np.cov(YRank, YShift)
    SRank = Cov[0, 1] / Cov[0, 0]
    return SRank

# Création de la fonction pour calculer l'erreur
def BootSRank(Y, X, I, B):
    """
    I est un indice de {1,...,p}
    X est une matrice de N lignes, chaque ligne a pour valeur (X1, X2, ..., Xp)
    Y est une la sortie
    B nombre d'iterations
    """
    err = []
    for b in range(B):
        sample = np.random.choice(a=len(Y),size=len(Y),replace=True)
        err.append(compSRank(Y[sample],X[sample,:],I))
    errstd = np.std(err)
    return errstd

In [232]:
Srank = err = []
for i in range(9):
    Srank.append(compSRank(Y,X,i))
err = []
for i in range(9):
    err.append(BootSRank(Y,X,i,100))

In [ ]:
plt.bar(pred,Srank,  width = 0.5)
plt.errorbar(pred,Srank, yerr=err, fmt="o", color="r")
plt.show()

## Metamodel-assisted analysis

We will fit and test the performance of a [SVM regression model](https://scikit-learn.org/stable/modules/svm.html). See the first training session.

In [ ]:
sv0 = make_pipeline(StandardScaler(), SVR(kernel='rbf')).fit(X,Y)
sv0

We will randomly generate *Nmc=1000* samples and estimate the sea surface elevation using the trained metamodel. Replace in the following: ####.

In [ ]:
np.random.seed(10)
Nmc = #### ## Number of random samples
XRand = np.random.uniform(size=(Nmc,9))
XRand
for i in range(9):
    XRand[:,i] = XRand[:,i]*(X[:,i].max() - X[:,i].min()) + X[:,i].min()
YRand= sv0.predict(XRand)

**QUESTION 7** Re-run the algorithm for computing the Sobol sensitivity indices. Could you comment on the values? Do the same analyse with *Nmc=2500*, then with *Nmc=10000*  

In [ ]:
Srank = []
for i in range(9):
    Srank.append(compSRank(YRand,XRand,i))
err = []
for i in range(9):
    err.append(BootSRank(Y,X,i,100))
plt.bar(pred,Srank,  width = 0.5)
plt.errorbar(pred,Srank, yerr=err, fmt="o", color="r")
plt.show()